# ISIC 2024 - Baseline（W12 進度用）
流程：讀資料 → EDA → 按病人切的 5-fold CV → LightGBM（只用 metadata）→ pAUC → 產生 `submission.csv`

**提交前**：右側 Settings 把 **Internet 關掉**，然後按 Submit。

In [ ]:
import glob, io, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from PIL import Image
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

KAGGLE_INPUT = '/kaggle/input'

def find_dir(fname):
    # Kaggle 的掛載路徑會變（例如多一層 competitions/），所以用搜尋的找檔案所在資料夾
    for depth in range(1, 4):
        hits = sorted(glob.glob(os.path.join(KAGGLE_INPUT, *['*'] * depth, fname)),
                      key=lambda p: ('isic-2024-challenge' not in p, p))  # 優先選比賽本身的資料
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(f'在 {KAGGLE_INPUT} 底下找不到 {fname}，請確認右側 Input 有加入')

DATA = find_dir('train-metadata.csv')
print('資料路徑:', DATA)
SEED = 42

## 1. 讀資料

In [ ]:
train = pd.read_csv(f'{DATA}/train-metadata.csv', low_memory=False)
test = pd.read_csv(f'{DATA}/test-metadata.csv')
print('train:', train.shape, ' test:', test.shape)
print(train['target'].value_counts())
print('惡性比例: %.4f%%' % (train['target'].mean() * 100))
print('病人數:', train['patient_id'].nunique())

## 2. EDA（W12 資料分析頁用）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train['target'].value_counts().plot.bar(ax=axes[0], logy=True, title='Class count (log scale)')

per_patient = train.groupby('patient_id').size()
per_patient.plot.hist(bins=50, ax=axes[1], title='Images per patient')

site_rate = train.groupby('anatom_site_general')['target'].mean().sort_values()
(site_rate * 100).plot.barh(ax=axes[2], title='Malignant rate (%) by site')

plt.tight_layout(); plt.show()

print(train.groupby('sex')['target'].agg(['count', 'mean']))
print(train['age_approx'].describe())

In [ ]:
# 看幾張圖：上排惡性、下排良性
def load_img(h5, isic_id):
    return Image.open(io.BytesIO(h5[isic_id][()]))

pos_ids = train.loc[train.target == 1, 'isic_id'].sample(5, random_state=SEED)
neg_ids = train.loc[train.target == 0, 'isic_id'].sample(5, random_state=SEED)
with h5py.File(f'{DATA}/train-image.hdf5', 'r') as h5:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for ax, i in zip(axes[0], pos_ids):
        ax.imshow(load_img(h5, i)); ax.set_title('malignant'); ax.axis('off')
    for ax, i in zip(axes[1], neg_ids):
        ax.imshow(load_img(h5, i)); ax.set_title('benign'); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. 評分函式：pAUC（TPR ≥ 80%）
Kaggle 官方公式，分數範圍 0 ~ 0.2

In [ ]:
def pauc_above_tpr(y_true, y_pred, min_tpr=0.80):
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = 1.0 - np.asarray(y_pred)
    max_fpr = 1 - min_tpr
    scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    return 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (scaled - 0.5)

## 4. 特徵
只用 train 和 test **都有**的欄位（train 有些欄位像診斷結果在 test 沒有，用了會資料洩漏）

In [ ]:
drop = {'isic_id', 'patient_id', 'target', 'image_type', 'attribution', 'copyright_license'}
common = [c for c in test.columns if c in train.columns and c not in drop]
cat_cols = [c for c in common if not pd.api.types.is_numeric_dtype(train[c])]
num_cols = [c for c in common if c not in cat_cols]

for c in cat_cols:
    cats = pd.Categorical(train[c]).categories
    train[c] = pd.Categorical(train[c], categories=cats)
    test[c] = pd.Categorical(test[c], categories=cats)

features = num_cols + cat_cols
print(len(features), 'raw features')
print('categorical:', cat_cols)

### 病人內標準化特徵（ugly duckling）
醫師判斷時會看「這顆跟同一個人其他痣比起來特不特別」。
每個數值特徵減掉該病人的平均、除以標準差；再加上該病人的病灶數。train 和 test 各自在自己的資料裡算。

In [ ]:
pn_cols = [c for c in num_cols if c not in ('age_approx', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z')]

def add_patient_feats(df):
    g = df.groupby('patient_id')[pn_cols]
    z = (df[pn_cols] - g.transform('mean')) / (g.transform('std') + 1e-6)
    z.columns = [c + '_pnorm' for c in pn_cols]
    df = pd.concat([df, z], axis=1)
    df['patient_n_lesions'] = df.groupby('patient_id')['isic_id'].transform('count')
    return df

train = add_patient_feats(train)
test = add_patient_feats(test)
features = features + [c + '_pnorm' for c in pn_cols] + ['patient_n_lesions']
print(len(features), 'features')

## 5. 5-fold CV（按 patient_id 切，同一個病人不會同時出現在訓練和驗證）

In [ ]:
params = dict(
    objective='binary', learning_rate=0.03, num_leaves=15, min_child_samples=40,
    colsample_bytree=0.5, subsample=0.8, subsample_freq=1, reg_lambda=5,
    n_estimators=200, random_state=SEED, verbose=-1,
)
NEG_RATIO = 50   # 處理不平衡：每個 fold 的訓練資料，良性只抽惡性的 50 倍

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(train))
models = []
for fold, (tr, va) in enumerate(sgkf.split(train, train['target'], train['patient_id'])):
    d = train.iloc[tr]
    pos = d[d['target'] == 1]
    neg = d[d['target'] == 0].sample(len(pos) * NEG_RATIO, random_state=SEED)
    d = pd.concat([pos, neg])
    m = lgb.LGBMClassifier(**params)
    m.fit(d[features], d['target'])
    oof[va] = m.predict_proba(train.iloc[va][features])[:, 1]
    print(f'fold {fold}: pAUC = {pauc_above_tpr(train.iloc[va]["target"], oof[va]):.4f}')
    models.append(m)

print(f'\nOOF pAUC = {pauc_above_tpr(train["target"], oof):.4f}   (上限 0.2)')

In [ ]:
# 特徵重要度（簡報可用）
imp = pd.Series(np.mean([m.feature_importances_ for m in models], axis=0), index=features)
imp.sort_values().tail(20).plot.barh(figsize=(8, 7), title='Top 20 feature importance')
plt.tight_layout(); plt.show()

### 實驗紀錄（本機用 ISIC 官網同一份訓練資料跑的 5-fold OOF pAUC）
| 設定 | pAUC |
|---|---|
| 大模型 + scale_pos_weight=10 | 0.031 |
| 大模型，不加權 | 0.129 |
| 小模型（num_leaves=15、正則化） | 0.156 |
| 小模型 + 良性抽樣 1:50 | 0.156 |
| **+ 病人內標準化特徵（目前版本）** | **0.160** |

重點：對這麼少的正樣本，`scale_pos_weight` 反而嚴重過擬合；模型要小、要正則化。

## 6. 產生 submission.csv

In [ ]:
test_pred = np.mean([m.predict_proba(test[features])[:, 1] for m in models], axis=0)
sub = pd.DataFrame({'isic_id': test['isic_id'], 'target': test_pred})
sub.to_csv('submission.csv', index=False)
sub.head()